<a href="https://colab.research.google.com/github/cpython-projects/python_da_06_11_25/blob/main/lesson_27_part_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
import plotly.express as px

In [2]:
DB_USER = "prog_academy_da_8936_user"
DB_PASS = "ih9DVSH5hmpm2DvvES0wIRFSnOrkJ2FZ"
DB_HOST = "dpg-d6g4hu0gjchc73d78ju0-a.oregon-postgres.render.com"
DB_PORT = "5432"
DB_NAME = "prog_academy_da_8936"

In [3]:
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

У нас є користувачі, які:

* встановлюють застосунок (`app_sessions`),
* переглядають товари (`product_views_log`),
* авторизуються (`devices_users_map`),
* здійснюють покупки (`orders_log`).

# Робота з датами у SQL
**Основні функції для роботи з датою при аналізі даних**

## `DATE_TRUNC`  
**обрізає дату або час до заданої одиниці виміру**, обнуляючи менш значущі компоненти.

### Загальний синтаксис:

```sql
DATE_TRUNC('одиниця_часу', timestamp)
```

---

### Для чого використовується:

1. **Групування у часі**
   Наприклад, по тижнях, місяцях, днях тощо.
2. **Округлення до початку години/дня і т.д.**
3. **Порівняння дат з потрібною точністю**
   (наприклад, всі події за конкретний місяць).

---

### Підтримувані одиниці:

* `millennium`, `century`, `decade`, `year`, `quarter`, `month`
* `week`, `day`, `hour`, `minute`, `second`

In [6]:
query = """
  SELECT DATE_TRUNC('year', first_seen) FROM app_sessions LIMIT 5;
"""

df = pd.read_sql_query(query, engine)
df

,date_trunc
0,2024-01-01 00:00:00+00:00
1,2024-01-01 00:00:00+00:00
2,2024-01-01 00:00:00+00:00
3,2024-01-01 00:00:00+00:00
4,2024-01-01 00:00:00+00:00


## `EXTRACT`
**витягує окрему частину дати або часу**, наприклад: рік, місяць, день, година і т.д.

### Синтаксис:

```sql
EXTRACT(одиниця_часу FROM дата_або_час)
```

---

### Використовувані одиниці:

* `YEAR`
* `MONTH`
* `DAY`
* `HOUR`
* `MINUTE`
* `SECOND`
* `DOW` (day of week: 0 = Sunday, 6 = Saturday)
* `DOY` (day of year)
* `WEEK`
* `QUARTER`

---

### Для чого застосовується:

1. **Фільтрація по частині дати**
2. **Створення часових групувань**

---

### Різниця з `DATE_TRUNC`:

|                | `EXTRACT`                     | `DATE_TRUNC`                                 |
| -------------- | ----------------------------- | -------------------------------------------- |
| Що робить      | Повертає одне число           | Обрізає менш значущі частини дати            |
| Тип результату | `numeric`                     | `timestamp`                                  |
| Приклад        | `EXTRACT(MONTH FROM d)` → `7` | `DATE_TRUNC('month', d)` → `'2025-07-01...'` |

In [8]:
query = """
SELECT EXTRACT(DOW FROM first_seen) FROM app_sessions LIMIT 5;
"""

df = pd.read_sql_query(query, engine)
df

,extract
0,4.0
1,1.0
2,4.0
3,5.0
4,3.0


## `CURRENT_DATE`, `CURRENT_TIMESTAMP` и `NOW()`
**поточна дата та/або час**

### Особенности

| Функція             | Що повертає                     | Тип даних   | Приклад                      |
| ------------------- | ------------------------------- | ----------- | ---------------------------- |
| `CURRENT_DATE`      | Лише поточну дату               | `DATE`      | `2025-07-14`                 |
| `CURRENT_TIMESTAMP` | Поточну дату **та** час         | `TIMESTAMP` | `2025-07-14 11:25:32.123456` |
| `NOW()`             | Те саме, що `CURRENT_TIMESTAMP` | `TIMESTAMP` | `2025-07-14 11:25:32.123456` |


In [9]:
query = """
SELECT CURRENT_DATE, CURRENT_TIMESTAMP, NOW();
"""

df = pd.read_sql_query(query, engine)
df

,current_date,current_timestamp,now
0,2026-03-03,2026-03-03 17:54:24.100077+00:00,2026-03-03 17:54:24.100077+00:00


## `INTERVAL`  
**додавання/віднімання часових проміжків** (дні, години, місяці, секунди і т.д.)

### 📌 Синтаксис:

```sql
SELECT CURRENT_DATE + INTERVAL '7 days';
```
---

### Одиниці:

* `second`, `minute`, `hour`
* `day`, `week`, `month`, `year`
* * комбінації, наприклад:

```sql
INTERVAL '2 days 3 hours 15 minutes'
```
---

### Особливості:

* `INTERVAL` сам по собі — не дата, це **часовий відрізок**
* Використовується з `DATE` і з `TIMESTAMP`
* В MySQL запис трохи інший:

```sql
-- MySQL стиль
SELECT NOW() + INTERVAL 1 DAY;
```
---

### В Pandas аналог:

```python
import pandas as pd

pd.Timestamp.now() + pd.Timedelta(days=3, hours=2)
```

In [12]:
query = """
SELECT CURRENT_DATE - INTERVAL '7 days';
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    result = result.fetchone()
    print(result)

(datetime.datetime(2026, 2, 24, 0, 0),)


In [ ]:
query = """
SELECT CURRENT_DATE + INTERVAL '7 month';
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    result = result.fetchone()
    print(result)

(datetime.datetime(2026, 3, 26, 0, 0),)


In [13]:
query = """
SELECT CURRENT_DATE + INTERVAL '1 month 2 years 6 days';
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    result = result.fetchone()
    print(result)

(datetime.datetime(2028, 4, 9, 0, 0),)


In [16]:
query = """
SELECT session_id, first_seen, first_seen + INTERVAL '7 day' AS tomorrow
FROM app_sessions
WHERE first_seen >= CURRENT_DATE - INTERVAL '630 day';
"""

df = pd.read_sql_query(query, engine)
df

,session_id,first_seen,tomorrow
0,S00017,2024-06-26,2024-07-03
1,S00020,2024-06-22,2024-06-29
2,S00028,2024-06-18,2024-06-25
3,S00035,2024-06-15,2024-06-22
4,S00041,2024-06-27,2024-07-04
5,S00043,2024-06-23,2024-06-30
6,S00063,2024-06-21,2024-06-28
7,S00070,2024-06-20,2024-06-27
8,S00074,2024-06-25,2024-07-02
9,S00102,2024-06-21,2024-06-28


## Коли застосовується робота з датами та часом

| Ціль                                      | Коли застосовуємо                                  | Що потрібно                      |
| ----------------------------------------- | -------------------------------------------------- | -------------------------------- |
| **1. Графіки по днях / тижнях / місяцях** | Динаміка показників у часі                         | Групування по даті               |
| **2. Сезонність**                         | Піки/спади у поведінці                             | Групування по дню тижня, місяцю  |
| **3. Повторюваність покупок**             | Як часто повертаються користувачі                  | Обчислення інтервалів між датами |
| **4. Retention**                          | Чи повертаються користувачі після першої взаємодії | Когорти за першою подією         |

In [24]:
query = """
SELECT first_seen, count(*) AS installs
FROM app_sessions
GROUP BY first_seen
ORDER BY first_seen desc
"""

df = pd.read_sql(query, engine)
df.set_index('first_seen', inplace=True)
df

,installs
first_seen,
2024-06-28,2
2024-06-27,1
2024-06-26,3
2024-06-25,2
2024-06-24,1
...,...
2024-01-05,3
2024-01-04,4
2024-01-03,3


In [25]:
fig = px.line(df)
fig.show()

In [29]:
query = """
SELECT EXTRACT('week' from order_time) as week, count(*) as orders_counts
FROM orders_log
GROUP BY week
ORDER BY week
"""

df = pd.read_sql(query, engine)
df

,week,orders_counts
0,1.0,110
1,2.0,99
2,3.0,105
3,4.0,95
4,5.0,106
5,6.0,111
6,7.0,104
7,8.0,92
8,9.0,100
9,10.0,98


In [35]:
query = """
SELECT EXTRACT(ISODOW from order_time) as order_day_of_week, Count(*), sum(total_uah), avg(total_uah)
FROM orders_log
GROUP BY order_day_of_week
ORDER BY order_day_of_week
"""

df = pd.read_sql(query, engine)
df

,order_day_of_week,count,sum,avg
0,1.0,393,611202.48,1555.222595
1,2.0,347,523238.60,1507.892219
2,3.0,381,606162.49,1590.977664
3,4.0,363,569434.62,1568.690413
4,5.0,365,590665.10,1618.260548
5,6.0,378,590484.23,1562.127593
6,7.0,332,504473.13,1519.497380


In [42]:
query = """
SELECT user_uuid, max(order_time) - min(order_time) as diff, count(*) as orders_count
FROM orders_log
GROUP BY user_uuid
HAVING max(order_time) - min(order_time) > 0
"""

df = pd.read_sql(query, engine)
df

,user_uuid,diff,orders_count
0,USR0107,60,5
1,USR0066,159,13
2,USR0106,160,14
3,USR0224,140,9
4,USR0138,154,14
...,...,...,...
294,USR0231,159,13
295,USR0008,173,15
296,USR0104,169,14
297,USR0023,156,13
